# 11e — Jeux globaux : la coordination avec information incomplète
**Navigation** : [<< 11-BayesianGames](GameTheory-11-BayesianGames-Python.ipynb) | [11c-Global-Games (sibling)](GameTheory-11c-Global-Games-Python.ipynb) | [Index](README.md)


**Autour de 11** — prolonge [11 · Jeux bayésiens](GameTheory-11-BayesianGames-Python.ipynb), [04 · Équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb) et le jumeau [11c — sélection d'équilibre par trembles](GameTheory-11c-Global-Games-Python.ipynb) (mêmes jeux globaux, autre question : ici la sélection robuste du **plus petit seuil**, là le modèle de trembles).

Un jeu de coordination 2×2 à information **complète** porte **deux** équilibres de Nash purs (les deux accords) plus un équilibre mixte instable : la **multiplicité** n'est pas tranchée. Le dispositif des **jeux globaux** (Carlsson & van Damme 1993 ; Morris & Shin 1998) donne à chaque joueur un **signal bruité** d'un fondamental commun `θ`.

**La question de ce carnet** : quel **seuil robuste** le bruit sélectionne-t-il, et vers quel équilibre ce seuil conduit-il quand le bruit disparaît ? Le plus petit seuil d'équilibre est unique par définition et, quand `σ → 0`, il sélectionne **l'équilibre efficace** dès que le fondamental le mérite. Plusieurs racines de `g_sym` peuvent toutefois coexister : le carnet les compte sans confondre robustesse et unicité globale.

> Cœur du carnet : intégration gaussienne en forme close + recherche de racine, avec une fenêtre adaptée au bruit. Le résultat est **falsifiable**.

La lecture suit une trajectoire en cinq gestes : compter les équilibres du jeu complet (section 1), regarder le canal dégradé qu'est le signal (section 2), calculer le plus petit seuil puis le voir converger vers `C` (section 3), lire la sélection qu'il opère — le mur autour de `θ = C` (section 3 toujours) — et relier le tout (section 4). Les trois grandeurs affichées en route — `p*`, `k*`, P(les deux jouent H) — sont les coordonnées du même récit : le bruit choisit l'équilibre.

In [1]:
import numpy as np
import math

# --- Loi normale standard, via la stdlib (pas besoin de scipy) ---
def phi(x):
    """Densité normale standard : phi(x) = exp(-x^2/2) / sqrt(2 pi)."""
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def Phi(x):
    """Fonction de répartition normale standard : Phi(x) = 0.5 (1 + erf(x/sqrt 2))."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# --- Paramètres du jeu (partagés par tout le carnet) ---
TAU2 = 1.0   # variance a priori du fondamental theta ~ N(0, TAU2)
C    = 1.0   # coût d'agir seul : payer H face a L rapporte -C

np.random.seed(42)   # reproductibilité des tirages (si simulation)
print("Imports OK  |  phi(0) =", round(phi(0), 4), " |  Phi(0) =", Phi(0), " |  C =", C)

Imports OK  |  phi(0) = 0.3989  |  Phi(0) = 0.5  |  C = 1.0


### Lecture du résultat — la forme close, sans `scipy`

Deux fonctions portent tout le calcul du carnet : `phi` (densité) et `Phi` (fonction de répartition) de la normale standard, **implémentées à la main** sur la stdlib (`math.exp`, `math.erf`) — pas de dépendance `scipy`. La ligne de contrôle le confirme : `phi(0) = 0.3989` (le pic de la cloche, `1/√(2π)`) et `Phi(0) = 0.5` (la symétrie : la moitié de la masse est à gauche de 0). Ces deux valeurs sont les **deux** points que toute implémentation correcte doit restituer ; c'est le contrôle de cohérence du reste du carnet, où l'utilité espérée de `H` s'écrit **en forme close** à partir de `phi` et `Phi` (section 3).

## 1. Le jeu de coordination complet — la multiplicité d'origine

Deux joueurs, chacun joue **H** (investir / entrer) ou **L** (s'abstenir). Il existe un **fondamental** `θ` (par ex. la rentabilité d'un projet) : jouer H rapporte `θ` **si l'autre joue aussi H**, mais coûte `C` si l'on joue H **seul**.

| Moi \ Adverse | **H** | **L** |
|---|---|---|
| **H** | `θ` | `−C` |
| **L** | `0` | `0` |

À **information complète** (chaque joueur connaît `θ`) :
- **(H,H)** est un équilibre de Nash **si `θ > 0`** (H est une meilleure réponse à H) ;
- **(L,L)** est **toujours** un équilibre (face à L, L bat H : `0 > −C`) ;
- l'**équilibre mixte** (chacun joue H avec probabilité `p* = C/(θ+C)`) existe quand `θ > 0`.

**Donc, pour `θ > 0` : deux équilibres purs + un mixte = multiplicité.** Le jeu seul ne dit pas lequel se réalisera.

Le mixte mérite un détour, car « instable » a un sens précis ici. À `θ = +0.5`, `p* = 0.6667` n'est pas une conduite qu'on observerait : c'est la fréquence qui rend l'adversaire **indifférent** — un joueur qui mixte exactement à `p*` face au même stratège gagne zéro en espérance, comme s'il jouait L. Baissez légèrement votre probabilité : l'autre a intérêt à jouer L tout court. Augmentez-la : H domine. Le mixte sépare donc deux attracteurs, et toute la question du carnet est de savoir lequel des deux le bruit retient — et à quelle frontière de `θ`.

In [2]:
# === Section 1 : le jeu de coordination complet (information complète) ===

def equilibres_purs(theta):
    """Liste des équilibres de Nash purs du jeu 2x2 de coordination."""
    ne = []
    if theta > 0:            # (H,H) : H meilleure réponse a H  <=>  theta > 0
        ne.append(("H", "H"))
    ne.append(("L", "L"))    # (L,L) : toujours un équilibre (0 > -C)
    return ne

def equilibre_mixte(theta):
    """Proba p* d'indifférence (mixte) ; None si hors de [0, 1]."""
    # Indifférence :  p* (theta) + (1 - p*) (-C) = 0   =>   p* = C / (theta + C)
    p = C / (theta + C)
    return p if 0.0 <= p <= 1.0 else None

for theta in (0.5, -0.5):
    print(f"\ntheta = {theta:+.1f}")
    print("  Nash purs   :", equilibres_purs(theta))
    pm = equilibre_mixte(theta)
    print("  Nash mixte p* :", (round(pm, 4) if pm is not None else "aucun (hors [0,1])"))
print("\n-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta <= 0 : (L,L) seul.")


theta = +0.5
  Nash purs   : [('H', 'H'), ('L', 'L')]
  Nash mixte p* : 0.6667

theta = -0.5
  Nash purs   : [('L', 'L')]
  Nash mixte p* : aucun (hors [0,1])

-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta <= 0 : (L,L) seul.


### Lecture — la multiplicité en nombres

La sortie sépare deux mondes en un seul paramètre. À `θ = +0.5`, le jeu porte **deux** équilibres purs, `('H', 'H')` et `('L', 'L')`, plus un mixte `p* = 0.6667` : chacun investit avec deux chances sur trois, et c'est exactement la fréquence qui rend l'autre indifférent — à `p*`, jouer H et jouer L rapportent tous deux zéro en espérance. À `θ = −0.5`, la liste ne contient plus que `('L', 'L')` : le gain commun d'un investissement réussi est négatif, H n'est jamais une meilleure réponse, et le mixte disparaît car `p* = C/(θ+C)` déborderait de `[0, 1]` — la fonction le signale en renvoyant `None`.

Retenez la mécanique du mixte : l'indifférence exige `p·θ − (1 − p)·C = 0`, d'où `p* = C/(θ+C)`. Quand `θ` tend vers 0 par valeurs positives, `p*` tend vers 1 — il faut être presque sûr que l'autre investit pour oser investir dans un projet à peine rentable. C'est cette fragilité de l'accord ('H', 'H'), qui ne survit que par certitude mutuelle, que le bruit va exploiter.

## 2. Le signal bruité et la stratégie à seuil

Personne ne connaît `θ` directement. Chaque joueur `i` observe un **signal bruité**

$$s_i = \theta + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \ \text{(indépendants).}$$

On restreint aux **stratégies à seuil** : jouer **H** si et seulement si `s_i ≥ k`. La décision se résume à une **frontière** `k` dans l'espace des signaux — la classe de stratégies naturelle quand l'information est un signal du fondamental.

Économiquement, `s = θ + ε` se lit : chaque joueur observe le fondamental à travers son propre canal dégradé — deux analystes voient la même économie avec des modèles et des données légèrement différents. La variance `σ²` mesure la qualité du canal ; restreindre aux stratégies à seuil dit seulement que la décision finale est binaire : au-dessus de `k`, le signal paraît assez bon pour tenter l'investissement. Toute la subtilité du jeu passera dans le choix de cette frontière.

In [3]:
# === Section 2 : le signal bruité + stratégie à seuil ===

def signaux(theta, sigma, n=1, rng=None):
    """n signaux bruités d'un même fondamental : s = theta + N(0, sigma^2)."""
    rng = rng or np.random.default_rng()
    return theta + rng.normal(0.0, sigma, size=n)

theta0, sigma0 = 0.5, 0.3
ex = signaux(theta0, sigma0, n=6, rng=np.random.default_rng(42))
print(f"theta = {theta0},  sigma = {sigma0}")
print("6 signaux tirés :", np.round(ex, 3))
print("-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.")

theta = 0.5,  sigma = 0.3
6 signaux tirés : [ 0.591  0.188  0.725  0.782 -0.085  0.109]
-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.


### Lecture — ce que chaque joueur voit

Six tirages pour un seul fondamental `θ = 0.5` : les signaux s'étalent de `−0.085` à `0.782`. Deux joueurs du **même** jeu peuvent ainsi voir des mondes opposés — l'un croit le projet nettement rentable (`0.782`, loin au-dessus du vrai `θ`), l'autre le croit déficitaire (`−0.085`, sous zéro alors que le fondamental est positif). Aucun des deux n'est « mal informé » au sens où une correction suffirait : le bruit `ε` est intrinsèque au canal d'observation.

C'est le déclic des jeux globaux. À information complète, la multiplicité s'accrochait à la connaissance commune de `θ` : chaque confiance dans l'accord (H, H) se validait elle-même. Ici cette connaissance n'existe plus — il ne reste qu'une **stratégie** à deviner chez l'autre, et la classe des stratégies à seuil `k` est la plus simple à coordonner. La question devient calculable : existe-t-il un seuil `k` tel que « jouer H si `s ≥ k` » soit un équilibre, et si plusieurs existent, lequel survit au bruit ?

## 3. Le plus petit seuil d'équilibre — et pourquoi il est robuste

Avec un **a priori** `θ ~ N(0, τ²)` et un signal gaussien, la croyance a posteriori d'un joueur de signal `s` est gaussienne : `θ | s ~ N(m, v)` avec

$$m = 	frac{v\,s}{\sigma^2}, \qquad v = \Big(	frac{1}{	au^2} + 	frac{1}{\sigma^2}\Big)^{-1}.$$

Face à un adversaire de seuil `k`, l'utilité espérée de **H** s'écrit **en forme close** (intégration gaussienne) :

$$\mathbb{E}[U_H \mid s] = (m+C)\,\Phi\!\Big(	frac{m-k}{w}\Big) + 	frac{v}{w}\,\varphi\!\Big(	frac{m-k}{w}\Big) - C, \qquad w = \sqrt{v + \sigma^2},$$

où `Φ` et `φ` sont la fonction de répartition et la densité de la normale standard. Un **seuil d'équilibre** `k*` est une valeur où un joueur de signal `s = k*` est **indifférent** face à un adversaire de même seuil : c'est une racine de `g(k) = U_H(k; k) = 0`. Le **plus petit** de ces seuils est l'équilibre **robuste** ; les racines supérieures ne sont pas la sélection étudiée ici.

**Le point clé** : ce plus petit seuil est bien défini et, quand le bruit disparaît, il converge vers le **coût d'agir seul** : `k* → C`. Cela ne prouve **pas** qu'il est la seule racine : la mesure suivante distingue sélection robuste et multiplicité résiduelle.

D'où viennent `m` et `v` ? De la règle de Bayes pour deux gaussiennes : l'a priori `θ ~ N(0, τ²)` et le signal `s` se combinent en une croyance `θ | s ~ N(m, v)` dont les **précisions** s'additionnent (`1/v = 1/τ² + 1/σ²`) et dont la moyenne penche vers le signal à raison de sa précision relative. L'utilité de H en découle par intégration gaussienne — la probabilité que l'autre reçoive un signal au-dessus de `k` sachant le mien — et tout se résout en la forme close affichée : l'équilibre entier tient dans deux fonctions spéciales, `Φ` et `φ`.

In [4]:
# === Section 3 : le plus petit seuil d'équilibre symétrique ===
# A priori theta ~ N(0, TAU2).  Signal : s = theta + eps, eps ~ N(0, sigma^2).

def post(s, sigma):
    """Croyance a posteriori theta | s : (moyenne m, variance v)."""
    v = 1.0 / (1.0 / TAU2 + 1.0 / sigma**2)
    m = v * s / sigma**2
    return m, v

def utilite_H(s, k, sigma):
    """Utilité espérée de H pour un joueur de signal s, face a un seuil adverse k."""
    m, v = post(s, sigma)
    w = math.sqrt(v + sigma**2)
    z = (m - k) / w
    return (m + C) * Phi(z) + (v / w) * phi(z) - C

def g_sym(k, sigma):
    """Indifférence symétrique : utilite_H d'un joueur de signal s = k face a un
    adversaire de MÊME seuil k. Les racines de g_sym sont les équilibres de seuil."""
    return utilite_H(k, k, sigma)

def seuil_equilibre(sigma, k_max=30.0, n=2000):
    """Le plus PETIT seuil d'équilibre (le robuste) : première racine de g_sym,
    trouvée par balayage depuis k=0 puis bisection."""
    g0 = g_sym(0.0, sigma)
    if g0 >= 0.0:
        return 0.0
    ks = np.linspace(0.0, k_max, n)
    for i in range(n - 1):
        a, b = ks[i], ks[i + 1]
        ga = g_sym(a, sigma)
        if ga * g_sym(b, sigma) <= 0.0:          # racine dans [a, b]
            for _ in range(80):
                mid = 0.5 * (a + b)
                if ga * g_sym(mid, sigma) <= 0.0:
                    b = mid
                else:
                    a, ga = mid, g_sym(mid, sigma)
            return 0.5 * (a + b)
    return float("nan")

def compte_racines(sigma, k_max=None, n=40000):
    """Racines sur une fenêtre adaptée au bruit, avec résolution fixe.

    La seconde racine s'éloigne en 1/sigma : une borne fixe masque les
    petites valeurs de sigma. 30/sigma couvre les six cas mesurés.
    """
    if k_max is None:
        k_max = 30.0 / sigma
    step = k_max / n
    previous = g_sym(0.0, sigma)
    count = 0
    for i in range(1, n + 1):
        current = g_sym(i * step, sigma)
        if previous * current < 0:
            count += 1
        previous = current
    return count

sigma = 0.3
kstar = seuil_equilibre(sigma)
print(f"sigma = {sigma}   ->   plus petit seuil d'équilibre (robuste)  k* = {kstar:.4f}")

sigma = 0.3   ->   plus petit seuil d'équilibre (robuste)  k* = 1.5908


### Lecture — pourquoi k* dépasse C

À `σ = 0.3`, le plus petit seuil d'équilibre vaut `k* = 1.5908`, nettement au-dessus de `C = 1`. La lecture naïve — « le seuil devrait se caler sur le coût d'agir seul » — est fausse tant que le bruit est fini : l'écart `k* − C ≈ 0.59` mesure la **double incertitude** d'un joueur. Il doute d'abord sur le fondamental (il ne voit que `s`, pas `θ`), puis sur le signal de l'autre (même à fondamental donné, l'adversaire observe un `s'` différent du sien). Pour oser jouer H, il faut être optimiste sur `θ` **et** sur la probabilité que l'autre reçoive aussi un signal au-dessus du seuil — deux marges de prudence qui s'additionnent et poussent la frontière au-dessus du cas sans bruit.

La forme close de `g_sym` encode tout ceci en une courbe : l'écart entre l'utilité de H et celle de L pour un joueur au seuil, positif à signal haut, qui décroît et recoupe zéro en `k*`. C'est cette plus petite racine que la cellule suivante traque pour toute une gamme de bruits.

In [5]:
# Multiplicité résiduelle : la borne de recherche suit le bruit.
print("Racines de g_sym sur [0, 30/sigma], avec 40000 intervalles :")
print(f"{'sigma':>7}  {'nb de seuils':>14}  {'plus petit k*':>15}")
for sigma in (0.3, 0.2, 0.1, 0.05, 0.01, 0.001):
    print(f"{sigma:>7.3f}  {compte_racines(sigma):>14}  {seuil_equilibre(sigma):>15.4f}")


Racines de g_sym sur [0, 30/sigma], avec 40000 intervalles :
  sigma    nb de seuils    plus petit k*
  0.300               2           1.5908
  0.200               2           1.2334


  0.100               2           1.0799


  0.050               2           1.0335


  0.010               2           1.0058


  0.001               2           1.0006


### Lecture — deux racines partout, une seule robuste

Deux régularités traversent le tableau. D'abord le **nombre de racines : 2** à chaque niveau de bruit — la multiplicité n'est pas éliminée par le bruit, elle est réorganisée. Ensuite la **dérive du plus petit seuil** : `k*` descend de `1.5908` (σ = 0.3) à `1.0006` (σ = 0.001), et la cible de la convergence est `C = 1`, pas `0`. Quand l'observation devient presque parfaite, le seuil d'équilibre se fixe sur le **coût d'agir seul** — le seul coût qui reste réel quand le fondamental devient observable.

Pourquoi garder « le plus petit » plutôt que n'importe quelle racine ? La théorie (Carlsson–van Damme 1993) montre que les racines supérieures ne résistent pas à une petite perturbation des croyances : un joueur dont le signal tombe juste sous le seuil sait que l'autre, face au même doute, jouerait L — et sa meilleure réponse bascule. La plus petite racine est le seul seuil où ce basculement local ne détruit pas l'équilibre : c'est la **sélection d'équilibre opérée par le bruit**, que le jumeau 11c démontre sur les trembles et que ce carnet se contente de mesurer.

In [6]:
# Probabilité (interne) de jouer H à l'équilibre : la part de joueurs qui jouent H.
#   s ~ N(theta, sigma^2)   =>   P(H) = P(s >= k*) = Phi((theta - k*) / sigma).
theta, sigma = 1.2, 0.3
kstar = seuil_equilibre(sigma)
p_H = Phi((theta - kstar) / sigma)
print(f"theta = {theta},  sigma = {sigma},  k* = {kstar:.4f}")
print(f"P(jouer H à l'équilibre) = {p_H:.4f}   ->  interne : le plus petit seuil fixe une part de population.")

theta = 1.2,  sigma = 0.3,  k* = 1.5908
P(jouer H à l'équilibre) = 0.0964   ->  interne : le plus petit seuil fixe une part de population.


### La sélection robuste (le résultat)

Au plus petit seuil, la part de joueurs qui jouent H est `P(H) = Φ((θ − k*)/σ)` — une valeur **interne** (ni 0 ni 1). D'autres racines peuvent exister, mais elles ne changent pas ce seuil robuste.

**Quand le bruit disparaît** (`σ → 0`), `k* → C` et les signaux convergent vers `θ`, donc :

- si **`θ > C`** : `P(les deux jouent H) → 1`  →  **l'équilibre EFFICACE (H,H) est sélectionné** ;
- si **`θ < C`** : `P(les deux jouent H) → 0`  →  l'équilibre (L,L) est sélectionné — et c'est un **échec de coordination** dès que `0 < θ < C` (l'équilibre efficace (H,H) existait, le bruit ne le sélectionne pas).

**Le bruit, au lieu d'embrouiller, fait le tri** autour du seuil `C` = coût d'agir seul. C'est le résultat de sélection robuste étudié dans ce carnet.

La borne `θ = C` est un énoncé de limite. À bruit fini, la sélection n'est pas encore tranchée : les colonnes de la cellule suivante montrent le phénomène en train de se faire, du « presque jamais » à bruit fort au « presque toujours » à bruit faible — et la transition entre les deux est raide. C'est cette raideur, mesurée, qui fait des jeux globaux un instrument prédictif : la frontière est fine et calculable.

In [7]:
# === La sélection robuste : sigma -> 0  =>  (H,H) si theta > C, sinon (L,L) ===
#   P(les deux jouent H) = [ Phi((theta - k*(sigma)) / sigma) ]^2   (signaux indépendants).

def prob_both_H(theta, sigma):
    k = seuil_equilibre(sigma)
    p = Phi((theta - k) / sigma)
    return p * p

sigmas = [0.3, 0.2, 0.1, 0.05, 0.01, 0.001]

print("theta = +1.2   (theta > C=1  ->  l'équilibre EFFICACE (H,H) doit être sélectionné)")
print(f"{'sigma':>7}  {'k*':>9}  {'P(les 2 jouent H)':>18}")
for s in sigmas:
    print(f"{s:>7.3f}  {seuil_equilibre(s):>9.4f}  {prob_both_H(1.2, s):>18.4f}")

print("\ntheta = +0.8   (theta < C=1  ->  échec de coordination : (L,L) sélectionné, bien que (H,H) soit efficace)")
print(f"{'sigma':>7}  {'k*':>9}  {'P(les 2 jouent H)':>18}")
for s in sigmas:
    print(f"{s:>7.3f}  {seuil_equilibre(s):>9.4f}  {prob_both_H(0.8, s):>18.4f}")

theta = +1.2   (theta > C=1  ->  l'équilibre EFFICACE (H,H) doit être sélectionné)
  sigma         k*   P(les 2 jouent H)
  0.300     1.5908              0.0093
  0.200     1.2334              0.1880
  0.100     1.0799              0.7836
  0.050     1.0335              0.9991
  0.010     1.0058              1.0000
  0.001     1.0006              1.0000

theta = +0.8   (theta < C=1  ->  échec de coordination : (L,L) sélectionné, bien que (H,H) soit efficace)
  sigma         k*   P(les 2 jouent H)
  0.300     1.5908              0.0000
  0.200     1.2334              0.0002
  0.100     1.0799              0.0000
  0.050     1.0335              0.0000
  0.010     1.0058              0.0000
  0.001     1.0006              0.0000


### Lecture — le mur de sélection

Pour `θ = 1.2` (au-dessus de `C`), la coordination sur (H, H) grimpe de `0.0093` à `σ = 0.3` — le seuil `k* = 1.5908` reste hors de portée de la plupart des signaux — puis perce : `0.1880` à `σ = 0.2`, `0.7836` à `σ = 0.1`, `0.9991` à `σ = 0.05`, et `1.0000` dès `σ = 0.01`. Pour `θ = 0.8` (sous `C`), la colonne reste collée à zéro, `0.0002` au maximum à `σ = 0.2` : sur un projet dont la rentabilité ne couvre pas le coût d'agir seul, personne n'investit, bruit ou pas.

La transition de la première colonne est raide : entre `σ = 0.2` et `σ = 0.05`, un facteur 4 de bruit fait passer la probabilité d'un cinquième à la quasi-certitude. C'est le **mur** — la sélection ne se débat pas, elle claque. Et son point de bascule n'est pas `θ = 0` (la frontière du jeu complet) mais `θ = C`. Entre les deux s'ouvre la zone des échecs de coordination : `0 < θ < C` rendrait (H, H) efficient à information complète, et le bruit le fait disparaître au profit de (L, L) — la crise de change qui n'a pas lieu parce que chacun doute que l'autre attaque.

## 4. Synthèse

| Information | Équilibres étudiés | Résultat quand `σ → 0` |
|---|---|---|
| **Complète** (`θ` connu) | **multiplicité** : (H,H), (L,L), mixte | — |
| **Incomplète** (signal `s = θ + ε`) | **plus petit seuil robuste** `k*` ; autres racines possibles | **(H,H) si `θ > C`**, sinon (L,L) |

Le bruit ne dégrade pas le jeu : il permet d'identifier un seuil robuste — au contraire de l'intuition. C'est ce qui rend les jeux globaux utiles pour étudier les crises de change, les paniques bancaires et la coordination sur un fondamental.

Un mot sur l'application canonique. Morris & Shin (1998) modélisent une attaque de change : des spéculateurs coordonnés renversent la parité, isolés ils perdent leur mise — exactement notre H/L avec, en fondamental, la solidité de la parité et, en coût `C`, celui d'une attaque infructueuse. La leçon du mur : une parité moyennement solide (`0 < θ < C`) peut tenir ou tomber selon la qualité de l'information publique — d'où l'attention des banques centrales à leurs communications, qui déplacent le bruit et donc l'équilibre sélectionné.

## 5. Exercices

Trois exercices pour consolider. Chaque fonction est un **stub** : complétez le corps. Les cellules s'exécutent telles quelles (elles renvoient `None` tant qu'incomplètes) — aucune erreur volontaire.

In [8]:
# Exercice 1 — La sélection robuste, côté L
def prob_both_L(theta, sigmas):
    """Pour chaque sigma, proba ex-post que les DEUX jouent L (cas theta < 0).

    Attendu : une liste de même longueur que sigmas ; quand sigma -> 0 et theta < 0,
    chaque valeur doit tendre vers 1.
    """
    # TODO : compléter
    # Indice : symétrique de prob_both_H. P(un joueur joue L) = 1 - Phi((theta - k*)/sigma),
    #         avec k* = seuil_equilibre(sigma) ; les signaux sont indépendants.
    result = None  # TODO etudiant
    return result

print(prob_both_L(-0.5, [0.5, 0.1, 0.01]))   # -> None tant que non complété

None


In [9]:
# Exercice 2 — Le seuil se fixe sur le coût d'agir seul
def seuils_en_fonction_de_sigma(sigmas):
    """Renvoie k*(sigma) pour chaque sigma.

    Attendu : on doit voir k* -> C (= 1 ici) quand sigma -> 0 (le seuil robuste se fixe
    sur le coût d'agir seul, pas sur 0).
    """
    # TODO : compléter
    # Indice : compréhension de liste — [seuil_equilibre(s) for s in sigmas]
    result = None  # TODO etudiant
    return result

print(seuils_en_fonction_de_sigma([1.0, 0.3, 0.05, 0.001]))   # -> None tant que non complété

None

In [10]:
# Exercice 3 — Une seule racine dans une fenêtre donnée ?
def est_unique(sigma, k_max=None):
    """True si g_sym a exactement une racine dans la fenêtre mesurée.

    Sans k_max explicite, utiliser la même fenêtre 30/sigma que la cellule
    de mesure : le résultat porte sur cette fenêtre, pas sur [0, infini).
    """
    # TODO : compléter
    # Indice : appeler compte_racines(sigma, k_max) et comparer à 1.
    result = None  # TODO etudiant
    return result

print(est_unique(0.05))   # -> None tant que non complété


None


### Synthèse — ce que les exercices verrouillent

Chaque stub vise **un** des faits mesurés ci-dessus, pas le carnet en général :

- **Exercice 1** (`prob_both_L`) — la sélection robuste **côté L** : symétrique de `prob_both_H`, il doit rendre des probabilités tendant vers `1` quand `σ → 0` et `θ < 0` (l'équilibre (L,L), **efficace** dans ce cas, devient quasi certain).
- **Exercice 2** (`seuils_en_fonction_de_sigma`) — la convergence `k* → C` : relire la colonne « plus petit k* » du balayage et vérifier que le seuil robuste se fige sur `C = 1`, **pas** sur `0`.
- **Exercice 3** (`est_unique`) — la **non**-unicité : `compte_racines` rend **2** sur la fenêtre mesurée, donc `est_unique` doit répondre `False` — le jeu global sélectionne le plus petit seuil **sans** que le jeu soit à une seule racine.

Les trois indices pointent vers les fonctions déjà mesurées ; il n'y a rien à réinventer, seulement à **réassembler** ce que le carnet a établi.

## Références

- **Carlsson, H., & van Damme, E. (1993).** « Global Games and Equilibrium Selection », *Econometrica* 61(5), 989–1018. — introduction du jeu global et de la sélection d'équilibre sous information bruitée.
- **Morris, S., & Shin, H. S. (1998).** « Unique Equilibrium in a Model of Self-Fulfilling Currency Attacks », *American Economic Review* 88(3), 587–597. — application canonique aux attaques de change.

**Place dans la série** : `11e`, « Autour de 11 » — avec `11` (jeux bayésiens), `11b` (Lean), `11c` (trembles) et `11d` (raffinements séquentiels).
